# Mini-TP 3 — Sirve tu modelo por gRPC

Operaciones de Aprendizaje Automático II · CEIA – FIUBA · Entrega individual

Expone por gRPC el mismo modelo que la Sesión 1 sirve por REST: XGBoost que predice si un
crimen reportado en Chicago (2024) termina en arresto.

## Qué se entrega

1. `tp3_grpc/scoring.proto` con mensajes de entrada y salida tipados.
2. Los stubs generados y un servidor que carga el modelo una sola vez.
3. Un cliente que llama al servicio (unary).
4. Un método de server-streaming que puntúa un lote.
5. La comparación de latencia contra el endpoint REST, con la reflexión al final.

## Cómo está organizado

El código vive en los `.py` del repo (`tp3_grpc/server.py`, `client.py`, `benchmark.py`) y está
cubierto por tests; este notebook los importa, los muestra y los ejecuta. De esa forma no hay dos
copias del mismo código que se desincronicen.

In [1]:
import logging
import os
from pathlib import Path

import arrest_model

# El notebook vive en tp3_grpc/, pero protoc y los imports esperan la raíz del repo.
REPO = Path(arrest_model.__file__).resolve().parent.parent
os.chdir(REPO)

# Los dos servicios dejan una línea de log por request, y acá corren en este mismo proceso: esas
# líneas irían a la salida de las celdas. Se silencian LOS DOS por dos razones: para no llenar
# el notebook de ruido, y porque durante el benchmark el costo de escribirlas se le cobraría al
# protocolo que quedara sin silenciar, y la comparación dejaría de ser justa.
for servicio in ("tp1_rest", "tp3_grpc"):
    logging.getLogger(servicio).setLevel(logging.WARNING)

print("Trabajando desde:", REPO)

Trabajando desde: /Users/belcattaneo/Documents/MIA/MLOPS2/mlops2-minitps


## 1. El modelo

Un único `model/model.pkl` con el `XGBClassifier` entrenado y los parámetros de codificación.
Lo comparten los tres mini-TPs: gRPC no reimplementa nada del modelo, solo lo expone.

In [2]:
from arrest_model.model import load_bundle

bundle = load_bundle()
metadata = bundle["metadata"]
print(f"{metadata['name']} v{metadata['version']} ({metadata['framework']})")
print("entradas crudas:", metadata["inputs"])
print("features del modelo:", metadata["features"])
print("MCC en test:", round(metadata["metrics"]["mcc"], 4))

chicago-arrest-xgboost v1 (xgboost 3.4.1)
entradas crudas: ['iucr', 'primary_type', 'location_description', 'date', 'latitude', 'longitude']
features del modelo: ['IUCR_freq', 'Primary_Type_freq', 'Location_Description_freq', 'Day_sin', 'X Coordinate_standardized', 'Y Coordinate_standardized', 'Distance Crime To Police Station_standardized']
MCC en test: 0.5811


## 2. El contrato `.proto`

El contrato declara campos nombrados y tipados, no un vector anónimo de números: son los mismos
6 campos crudos que valida la API REST. `date` viaja como `google.protobuf.Timestamp`, es decir
un instante absoluto; el servidor lo convierte a hora de Chicago, que es como se entrenó la
feature `Day_sin`.

In [3]:
from IPython.display import Code, display

display(Code(filename="tp3_grpc/scoring.proto"))

// Contrato del servicio de scoring por gRPC.
//
// Tipa los mismos 6 campos crudos que el contrato REST (arrest_model.schemas.CrimeReport):
// el servidor aplica la misma validación y la misma codificación antes de puntuar.
// Los stubs se regeneran con `make grpc-stubs`; los *_pb2*.py no se editan a mano.

syntax = "proto3";

package arrest;

import "google/protobuf/timestamp.proto";

// Reporte crudo de un crimen en Chicago.
message CrimeReport {
  string iucr = 1;                          // 3 dígitos y un dígito o letra, ej. "1310"
  string primary_type = 2;                  // uno de los 31 tipos vistos en train
  optional string location_description = 3; // si falta, se codifica como UNKNOWN
  google.protobuf.Timestamp date = 4;       // instante UTC; el servidor lo pasa a hora de Chicago
  optional double latitude = 5;             // optional para distinguir "falta" de "vale 0"
  optional double longitude = 6;
}

// Lote de reportes para puntuar en una sola llamada.
message CrimeBatch {
  repeated CrimeReport items = 1;
}

// Predicción y versión del modelo que la produjo.
message Prediction {
  int32 arrest = 1;
  double probability = 2;
  string model_name = 3;
  int32 model_version = 4;
}

service ArrestScoring {
  // Unary: un reporte, una predicción.
  rpc Predict(CrimeReport) returns (Prediction);

  // Server streaming: un lote, N predicciones en el mismo orden que los items.
  rpc PredictStream(CrimeBatch) returns (stream Prediction);
}

## 3. Genera los stubs

`protoc` lee el `.proto` y genera dos archivos: `scoring_pb2.py` con las clases de mensajes y
`scoring_pb2_grpc.py` con el stub del cliente y la clase base del servidor. No se editan a mano:
se regeneran con `make grpc-stubs`, y un test compara los versionados contra el `.proto` para
detectar que alguien lo haya cambiado sin regenerar.

In [4]:
import subprocess
import sys

subprocess.run(
    [sys.executable, "-m", "grpc_tools.protoc", "-I", ".",
     "--python_out=.", "--grpc_python_out=.", "tp3_grpc/scoring.proto"],
    check=True,
)
print("generados:", sorted(p.name for p in Path("tp3_grpc").glob("scoring_pb2*.py")))

generados: ['scoring_pb2.py', 'scoring_pb2_grpc.py']


## 4. El servidor

El modelo se carga una sola vez al arrancar (en `serve`) y queda en el servicer; cada RPC solo
valida y puntúa. La validación reusa el `CrimeReport` de la API REST, así que un payload inválido
da el mismo error en los dos protocolos: acá como `INVALID_ARGUMENT`.

In [5]:
display(Code(filename="tp3_grpc/server.py", language="python"))

"""Mini-TP 3: servidor gRPC que sirve el modelo de arrestos de Chicago.

Mismo núcleo que la API REST: valida con `CrimeReport` y puntúa con `predict`. Lo único
propio de acá es traducir entre los mensajes protobuf y ese núcleo.
"""

import logging
import time
from collections.abc import Callable, Iterator, Sequence
from concurrent import futures
from datetime import UTC
from typing import Any

import grpc
from pydantic import ValidationError

from arrest_model.model import load_bundle, predict
from arrest_model.schemas import CrimeReport, PredictionOut
from tp3_grpc import scoring_pb2, scoring_pb2_grpc

logger = logging.getLogger("tp3_grpc")

# Acá no hay uvicorn que configure el logging: sin esto los INFO se descartan en el contenedor.
logging.basicConfig(level=logging.INFO, format="%(levelname)s:     %(message)s")


def report_from_proto(message: scoring_pb2.CrimeReport) -> dict[str, Any]:
    """Pasa el mensaje a un dict con **solo los campos presentes**.

    Lo ausente se omite en vez de viajar en cero: así un campo que falta da error de
    validación y no una predicción sobre un dato inventado.
    """
    report: dict[str, Any] = {"iucr": message.iucr, "primary_type": message.primary_type}
    if message.HasField("location_description"):
        report["location_description"] = message.location_description
    if message.HasField("date"):
        # ToDatetime() devuelve UTC sin zona; se la ponemos para que CrimeReport la convierta
        # a hora de Chicago, que es como se entrenó el Day_sin.
        report["date"] = message.date.ToDatetime().replace(tzinfo=UTC)
    for field in ("latitude", "longitude"):
        if message.HasField(field):
            report[field] = getattr(message, field)
    return report


def validation_detail(error: ValidationError) -> str:
    """Resume los errores nombrando el campo que falló, sin exponer nada interno."""
    return "; ".join(
        f"{'.'.join(str(part) for part in item['loc'])}: {item['msg']}" for item in error.errors()
    )


def prediction_to_proto(prediction: PredictionOut) -> scoring_pb2.Prediction:
    """Pasa la predicción del núcleo al mensaje del contrato gRPC."""
    return scoring_pb2.Prediction(
        arrest=prediction.arrest,
        probability=prediction.probability,
        model_name=prediction.model_name,
        model_version=prediction.model_version,
    )


def validated_reports(
    messages: Sequence[scoring_pb2.CrimeReport], context: grpc.ServicerContext, *, indexed: bool
) -> list[CrimeReport]:
    """Valida **todos** los mensajes antes de que se puntúe ninguno.

    Con `indexed`, el detalle del error dice qué item del lote falló: en un stream, si no
    se validara todo primero, el cliente recibiría predicciones y recién después el error.
    """
    reports = []
    for index, message in enumerate(messages):
        try:
            reports.append(CrimeReport.model_validate(report_from_proto(message)))
        except ValidationError as exc:
            prefix = f"item {index}: " if indexed else ""
            context.abort(grpc.StatusCode.INVALID_ARGUMENT, prefix + validation_detail(exc))
    return reports


class ArrestScoringServicer(scoring_pb2_grpc.ArrestScoringServicer):
    """Sirve un modelo ya cargado; la carga es responsabilidad de `serve`."""

    def __init__(self, bundle: dict[str, Any]) -> None:
        self.bundle = bundle

    def Predict(  # noqa: N802 (el nombre lo fija el .proto)
        self, request: scoring_pb2.CrimeReport, context: grpc.ServicerContext
    ) -> scoring_pb2.Prediction:
        """Unary: valida un reporte y devuelve su predicción."""
        reports = validated_reports([request], context, indexed=False)
        return prediction_to_proto(predict(self.bundle, reports)[0])

    def PredictStream(  # noqa: N802 (el nombre lo fija el .proto)
        self, request: scoring_pb2.CrimeBatch, context: grpc.ServicerContext
    ) -> Iterator[scoring_pb2.Prediction]:
        """Server streaming: valida el lote enter

In [6]:
from tp3_grpc.server import serve

# port=0 -> el sistema elige uno libre. serve() arranca y devuelve sin bloquear.
servidor, puerto = serve(port=0, bundle=bundle)
print("servidor gRPC escuchando en el puerto", puerto)

servidor gRPC escuchando en el puerto 59508


## 5. El cliente (unary)

`stub.Predict(...)` parece una llamada local, pero serializa el mensaje, lo manda por HTTP/2 y
deserializa la respuesta. Eso es el *remote procedure call*.

In [7]:
import grpc

from tp3_grpc import scoring_pb2, scoring_pb2_grpc
from tp3_grpc.client import crime_report

canal = grpc.insecure_channel(f"127.0.0.1:{puerto}")
stub = scoring_pb2_grpc.ArrestScoringStub(canal)

prediccion = stub.Predict(crime_report())
print("arrest:", prediccion.arrest, "| probability:", prediccion.probability)
print("modelo:", prediccion.model_name, "v" + str(prediccion.model_version))

arrest: 0 | probability: 0.06843266636133194
modelo: chicago-arrest-xgboost v1


In [8]:
# Un reporte inválido no llega al modelo: el contrato lo corta antes.
try:
    stub.Predict(crime_report(primary_type="BANANA"))
except grpc.RpcError as error:
    print(error.code().name, "->", error.details()[:90], "...")

INVALID_ARGUMENT -> primary_type: Input should be 'ARSON', 'ASSAULT', 'BATTERY', 'BURGLARY', 'CONCEALED CARRY  ...


## 6. Streaming

`PredictStream` recibe un lote y devuelve N respuestas, una por item. El servidor valida todo el
lote antes de emitir nada (si no, el cliente recibiría predicciones y recién después el error) y
llama al modelo una sola vez para las N filas.

In [9]:
lote = scoring_pb2.CrimeBatch(
    items=[crime_report(), crime_report(primary_type="THEFT")] * 5
)
for i, p in enumerate(stub.PredictStream(lote), start=1):
    print(f"[{i:2}] arrest={p.arrest} probability={p.probability:.6f}")

[ 1] arrest=0 probability=0.068433
[ 2] arrest=1 probability=0.765073
[ 3] arrest=0 probability=0.068433
[ 4] arrest=1 probability=0.765073
[ 5] arrest=0 probability=0.068433
[ 6] arrest=1 probability=0.765073
[ 7] arrest=0 probability=0.068433
[ 8] arrest=1 probability=0.765073
[ 9] arrest=0 probability=0.068433
[10] arrest=1 probability=0.765073


In [10]:
import time

# ¿Las predicciones llegan de a una o todas juntas al final?
grande = scoring_pb2.CrimeBatch(items=[crime_report()] * 100)
inicio = time.perf_counter()
primera = None
for i, _ in enumerate(stub.PredictStream(grande)):
    if i == 0:
        primera = (time.perf_counter() - inicio) * 1000
ultima = (time.perf_counter() - inicio) * 1000
print(f"lote de 100 -> primera a los {primera:.2f} ms, última a los {ultima:.2f} ms")

lote de 100 -> primera a los 3.55 ms, última a los 7.41 ms


## 7. Comparación de latencia gRPC vs REST

Se midió la misma predicción contra las dos APIs. Las dos sirven el mismo modelo y hacen el mismo
trabajo por llamada: validar el reporte, codificar las 7 features y predecir. Lo único que cambia
es el protocolo.

Se midieron cuatro cohortes: las dos APIs corriendo como proceso en la máquina (`rest-local` y
`grpc-local`) y las dos corriendo en contenedor (`rest-dockerizado` y `grpc-dockerizado`). A cada
una se le tomaron dos pruebas: una llamada suelta y un lote de 100 reportes. De REST se miden
además dos formas de conectarse, abriendo la conexión en cada llamada y reutilizándola, porque
gRPC siempre reutiliza el canal y la comparación pareja es contra esa segunda forma.

Cada cohorte se midió tres veces, con 300 repeticiones por medición, porque con una sola no se
distingue un efecto real del ruido. La tabla muestra la mediana de esas tres corridas; los
valores crudos quedaron guardados en `tp3_grpc/latencias.json`.

La medición se corre fuera de este notebook: las cohortes en contenedor no se pueden medir desde
adentro, y con los servicios corriendo en este mismo proceso el trabajo que hacen por request
competiría con el cliente que mide.

El detalle del método y la lectura de los números están en `tp3_grpc/latencias.md`.

In [11]:
from tp3_grpc.benchmark import median_table

print(median_table("tp3_grpc/latencias.json"))

| cohorte | prueba | variante | mediana (ms) |
|---|---|---|---|
| `rest-local` | una llamada | sin sesión | 2.78 |
|  | una llamada | keep-alive | 2.66 |
|  | lote de 100 | 100 llamadas sueltas | 255.83 |
|  | lote de 100 | 1 llamada al batch | 3.32 |
| `rest-dockerizado` | una llamada | sin sesión | 5.63 |
|  | una llamada | keep-alive | 4.94 |
|  | lote de 100 | 100 llamadas sueltas | 507.65 |
|  | lote de 100 | 1 llamada al batch | 6.92 |
| `grpc-local` | una llamada | canal reusado | 1.85 |
|  | lote de 100 | 1 PredictStream | 6.46 |
| `grpc-dockerizado` | una llamada | canal reusado | 2.81 |
|  | lote de 100 | 1 PredictStream | 13.46 |


Para rehacer la medición, desde la raíz del repo se levantan los dos servicios y se
corre `make grpc-bench`: locales con `uvicorn tp1_rest.app:app` y `python -m tp3_grpc.server`, o
en contenedor con `make rest-up` y `make grpc-up`.

## 8. Reflexión

Qué diferencia de latencia observé: gRPC dio entre 1.4x y 1.6x más rápido que REST con
keep-alive corriendo local, y hasta 1.9x con los dos servicios en contenedor. Es una diferencia
real pero moderada, porque las dos APIs hacen el mismo trabajo por llamada —validar, codificar y
predecir— y el transporte es apenas una fracción de ese tiempo. Lo que más movió la aguja no fue
el protocolo sino agrupar: mandar 100 reportes en una sola llamada rindió entre 47 y 83 veces más
que hacer 100 llamadas sueltas, y ahí el lote por REST incluso le ganó al streaming de gRPC.

Cuándo usaría cada uno: REST de cara afuera, porque lo habla el navegador, se prueba con curl y
cualquier consumidor lo entiende sin herramientas extra. gRPC para el tráfico interno entre
servicios, donde rinde la conexión persistente: al dockerizar, REST sin sesión se degradó un 102%
y gRPC solo un 52%, porque reutiliza una conexión en vez de abrir una nueva en cada llamada.

Qué cuesta gRPC: el mensaje pesa 2.7 veces menos que el JSON equivalente, pero se paga en
tooling. Hay que generar y versionar los stubs y acordarse de regenerarlos al tocar el contrato,
no se puede probar con curl ni desde el navegador, y los errores son códigos de gRPC que hay que
mapear en vez de status HTTP. Para un endpoint público ese costo no se justifica.

## 9. Limpieza

In [12]:
canal.close()
servidor.stop(0)
print("servidor gRPC detenido")

servidor gRPC detenido
